# CNN Image Classification — Two External Datasets

This notebook applies CNN image classification to two external image datasets:

1. **Cats vs Dogs** — binary classification.
2. **Rock Paper Scissors** — 3-class classification.

The datasets are downloaded directly from public online sources during notebook execution, so no built-in TensorFlow dataset is being used.

### Workflow
- Download external dataset
- Load images from folders
- Preprocess and normalize
- Build CNN
- Train and validate
- Evaluate on test/validation data
- Plot Accuracy and Loss
- Confusion Matrix
- Classification Report
- Sample Prediction


# Dataset 1 — Cats vs Dogs

External dataset source:
https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip

The dataset contains separate `train` and `validation` folders with `cats` and `dogs` classes.


In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Rescaling, RandomFlip, RandomRotation
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow version:", tf.__version__)


In [ ]:
URL = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"

zip_path = tf.keras.utils.get_file(
    "cats_and_dogs_filtered.zip",
    origin=URL,
    extract=True
)

BASE_DIR = os.path.join(os.path.dirname(zip_path), "cats_and_dogs_filtered")
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR = os.path.join(BASE_DIR, "validation")

print("Dataset path:", BASE_DIR)
print("Train path:", TRAIN_DIR)
print("Validation path:", VAL_DIR)


In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode="binary"
)

class_names1 = train_ds.class_names
print("Classes:", class_names1)


In [ ]:
plt.figure(figsize=(10, 6))

for images, labels in train_ds.take(1):
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names1[int(labels[i].numpy()[0])])
        plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


In [ ]:
model1 = Sequential([
    Input(shape=(128, 128, 3)),

    Rescaling(1./255),

    RandomFlip("horizontal"),
    RandomRotation(0.1),

    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D(),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),

    Dense(1, activation="sigmoid")
])

model1.summary()


In [ ]:
model1.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history1 = model1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)


In [ ]:
val_loss1, val_accuracy1 = model1.evaluate(val_ds, verbose=0)

print("Cats vs Dogs Validation Loss:", val_loss1)
print("Cats vs Dogs Validation Accuracy:", val_accuracy1)


In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history1.history["accuracy"], label="Train")
plt.plot(history1.history["val_accuracy"], label="Validation")
plt.title("Cats vs Dogs Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history1.history["loss"], label="Train")
plt.plot(history1.history["val_loss"], label="Validation")
plt.title("Cats vs Dogs Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
y_true1 = []
y_pred1 = []

for images, labels in val_ds:
    predictions = model1.predict(images, verbose=0)
    predictions = (predictions >= 0.5).astype(int).flatten()

    y_true1.extend(labels.numpy().astype(int).flatten())
    y_pred1.extend(predictions)

cm1 = confusion_matrix(y_true1, y_pred1)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm1,
    annot=True,
    fmt="d",
    xticklabels=class_names1,
    yticklabels=class_names1
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Cats vs Dogs Confusion Matrix")
plt.show()

print(classification_report(
    y_true1,
    y_pred1,
    target_names=class_names1
))


In [ ]:
for images, labels in val_ds.take(1):
    sample_image = images[0]
    actual_label = int(labels[0].numpy())

prediction = model1.predict(
    tf.expand_dims(sample_image, axis=0),
    verbose=0
)[0][0]

predicted_label = int(prediction >= 0.5)

plt.figure(figsize=(5, 5))
plt.imshow(sample_image.numpy().astype("uint8"))
plt.title(
    f"Actual: {class_names1[actual_label]}\n"
    f"Predicted: {class_names1[predicted_label]}"
)
plt.axis("off")
plt.show()


# Dataset 2 — Rock Paper Scissors

External dataset sources:
- Training: https://storage.googleapis.com/download.tensorflow.org/data/rps.zip
- Test: https://storage.googleapis.com/download.tensorflow.org/data/rps-test-set.zip

This dataset has three classes: rock, paper, and scissors.


In [ ]:
RPS_TRAIN_URL = "https://storage.googleapis.com/download.tensorflow.org/data/rps.zip"
RPS_TEST_URL = "https://storage.googleapis.com/download.tensorflow.org/data/rps-test-set.zip"

rps_train_zip = tf.keras.utils.get_file(
    "rps.zip",
    origin=RPS_TRAIN_URL,
    extract=True
)

rps_test_zip = tf.keras.utils.get_file(
    "rps-test-set.zip",
    origin=RPS_TEST_URL,
    extract=True
)

RPS_BASE = os.path.dirname(rps_train_zip)
RPS_TRAIN_DIR = os.path.join(RPS_BASE, "rps")
RPS_TEST_DIR = os.path.join(RPS_BASE, "rps-test-set")

print("RPS train path:", RPS_TRAIN_DIR)
print("RPS test path:", RPS_TEST_DIR)


In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds2 = tf.keras.utils.image_dataset_from_directory(
    RPS_TRAIN_DIR,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

test_ds2 = tf.keras.utils.image_dataset_from_directory(
    RPS_TEST_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

class_names2 = train_ds2.class_names

print("Classes:", class_names2)


In [ ]:
plt.figure(figsize=(10, 6))

for images, labels in train_ds2.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names2[int(labels[i].numpy())])
        plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
train_ds2 = train_ds2.prefetch(buffer_size=AUTOTUNE)
test_ds2 = test_ds2.prefetch(buffer_size=AUTOTUNE)


In [ ]:
model2 = Sequential([
    Input(shape=(128, 128, 3)),

    Rescaling(1./255),

    RandomFlip("horizontal"),
    RandomRotation(0.1),

    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D(),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),

    Dense(3, activation="softmax")
])

model2.summary()


In [ ]:
model2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history2 = model2.fit(
    train_ds2,
    validation_data=test_ds2,
    epochs=10
)


In [ ]:
test_loss2, test_accuracy2 = model2.evaluate(test_ds2, verbose=0)

print("Rock Paper Scissors Test Loss:", test_loss2)
print("Rock Paper Scissors Test Accuracy:", test_accuracy2)


In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history2.history["accuracy"], label="Train")
plt.plot(history2.history["val_accuracy"], label="Validation")
plt.title("Rock Paper Scissors Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history2.history["loss"], label="Train")
plt.plot(history2.history["val_loss"], label="Validation")
plt.title("Rock Paper Scissors Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
y_true2 = []
y_pred2 = []

for images, labels in test_ds2:
    predictions = model2.predict(images, verbose=0)
    predictions = np.argmax(predictions, axis=1)

    y_true2.extend(labels.numpy())
    y_pred2.extend(predictions)

cm2 = confusion_matrix(y_true2, y_pred2)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm2,
    annot=True,
    fmt="d",
    xticklabels=class_names2,
    yticklabels=class_names2
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Rock Paper Scissors Confusion Matrix")
plt.show()

print(classification_report(
    y_true2,
    y_pred2,
    target_names=class_names2
))


In [ ]:
for images, labels in test_ds2.take(1):
    sample_image2 = images[0]
    actual_label2 = int(labels[0].numpy())

prediction2 = model2.predict(
    tf.expand_dims(sample_image2, axis=0),
    verbose=0
)

predicted_label2 = int(np.argmax(prediction2))

plt.figure(figsize=(5, 5))
plt.imshow(sample_image2.numpy().astype("uint8"))
plt.title(
    f"Actual: {class_names2[actual_label2]}\n"
    f"Predicted: {class_names2[predicted_label2]}"
)
plt.axis("off")
plt.show()


# Conclusion

Two external image datasets were used successfully with CNN models:

- **Cats vs Dogs:** binary image classification.
- **Rock Paper Scissors:** multi-class image classification.

For both datasets, the notebook includes preprocessing, CNN architecture, training, evaluation, accuracy/loss plots, confusion matrix, classification report, and prediction.
